# Train EfficientDet-D4

This notebook trains an EfficientDet-D4 object detector on the saved merged NIH + VinBig train/validation split. It keeps all available samples from `04_merged_train_val_split_1024.csv`, then handles class imbalance during training with focal loss, class weights, stronger rare-class augmentation, and moderate weighted sampling.

Trace points to watch while running:

1. Configuration defines the dataset paths, model size, rare classes, and imbalance knobs.
2. Annotation loading verifies the split CSV schema, filters invalid boxes, and computes class weights.
3. Dataset code loads images, rescales boxes if needed, and applies rare-class augmentation only during training.
4. DataLoader code keeps all samples but uses weighted sampling to show rare-class images more often.
5. Model code creates `tf_efficientdet_d4` and configures focal-loss/class-weight hooks when exposed by `effdet`.
6. Training code applies a batch-level class-weight multiplier so class weights still matter even if the installed `effdet` loss does not expose native class-weight fields.

## 1. Environment

Install the local training dependencies before running the notebook. Your machine has an NVIDIA GPU, so this notebook uses the CUDA-enabled PyTorch wheel instead of the CPU-only wheel. Restart the kernel after package installation so imports resolve cleanly.

In [1]:
# Notebook guide: install the EfficientDet-D4 training dependencies for an NVIDIA GPU.
# Your previous PyTorch install was CPU-only (`torch ... +cpu`), so CUDA was unavailable.
# Run this cell once, then restart the kernel before continuing.

%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install -U effdet timm pandas pillow tqdm pyyaml mlflow

Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0
Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://download.pytorch.org/whl/cu128
  Obtaining dependency information for torch from https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Obtaining dependency information for torchvision from https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (5.6 kB)
  Obtaining dependency information for torchaudio from https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
   ---------------------------------------- 2.8/2.8 GB 998.8 kB/s eta 0:00:00
   ------------------

  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl (9.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Notebook guide: shared imports for training, augmentation, sampling, and experiment logging.
# `effdet` provides EfficientDet-D4; Pillow handles image augmentations; PyTorch handles training.

from __future__ import annotations

import random
from collections import defaultdict
from pathlib import Path

import mlflow
import pandas as pd
import torch
from effdet import create_model
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import functional as F
from tqdm.auto import tqdm

In [3]:
# Notebook guide: verify that this kernel can see your NVIDIA GPU before training.
# Expected after installing CUDA PyTorch: CUDA available = True, device = cuda.

print("Python torch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not visible to PyTorch. Restart the kernel after running the install cell above.")

Python torch version: 2.11.0+cu128
PyTorch CUDA build: 12.8
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti


## 2. Configuration

Set the training paths, model hyperparameters, and imbalance controls in one place. The rare-class list targets the classes with the lowest annotation counts: Consolidation, Atelectasis, and Pneumothorax.

In [4]:
# Notebook guide: define project paths, model settings, and imbalance strategy knobs.
# Keep `IMAGE_SIZE = 1024` because the prepared boxes are already transformed into 1024-space.

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "ML":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "notebooks" / "data"
ANNOTATION_CSV = DATA_DIR / "processed" / "04_merged_train_val_split_1024.csv"
NIH_IMAGE_DIR = DATA_DIR / "raw" / "RAW_NIH" / "images"
VINBIG_IMAGE_DIR = DATA_DIR / "raw" / "RAW_VINBIG" / "images"
IMAGE_DIRS = {"NIH": NIH_IMAGE_DIR, "VinBig": VINBIG_IMAGE_DIR}
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "models" / "efficientdet_d4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 1024
BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Keep 0 on Windows/Jupyter unless you know multiprocessing is stable.

# Imbalance controls: keep all samples, but lift rare classes during training.
RARE_CLASS_NAMES = {"Consolidation", "Atelectasis", "Pneumothorax"}
CLASS_WEIGHT_POWER = 0.5  # sqrt inverse-frequency weights; less aggressive than raw inverse freq.
SAMPLER_WEIGHT_POWER = 0.5  # moderate weighted sampling, not full rare-class oversampling.
FOCAL_LOSS_GAMMA = 2.0
FOCAL_LOSS_ALPHA = 0.25
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [5]:
# Notebook guide: seed random generators so train/validation splits and sampler order are reproducible.
# CuDNN benchmarking stays enabled for speed because EfficientDet-D4 uses fixed-size 1024 inputs.

def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

## 3. Load Prepared Annotations

Load the saved merged train/validation split CSV, validate that required columns are present, remove invalid boxes, and compute class weights from the actual annotation counts. This stage does not create a new split and does not drop valid samples for balancing.

In [6]:
# Notebook guide: load and validate the saved merged train/val split annotations.
# This cell keeps all valid boxes, clips coordinates into image bounds, and computes class weights.

df = pd.read_csv(ANNOTATION_CSV)
df = df.rename(columns={"class_name": "label_name"})
required_columns = {"source", "image_id", "label_name", "target_class_id", "split", "x_min", "y_min", "x_max", "y_max"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=list(required_columns)).copy()
df["source"] = df["source"].astype(str)
df["image_id"] = df["image_id"].astype(str)
df["split"] = df["split"].astype(str).str.lower()
df["image_key"] = df["source"] + "::" + df["image_id"]
for col in ["x_min", "y_min", "x_max", "y_max"]:
    df[col] = df[col].astype(float).clip(0, IMAGE_SIZE)
df = df[(df["x_max"] > df["x_min"]) & (df["y_max"] > df["y_min"])]
df["target_class_id"] = df["target_class_id"].astype(int)

label_map = (
    df[["target_class_id", "label_name"]]
    .drop_duplicates()
    .sort_values("target_class_id")
    .set_index("target_class_id")["label_name"]
    .to_dict()
)
NUM_CLASSES = len(label_map)

class_counts = df.groupby("target_class_id").size().sort_index()
class_weights = (class_counts.max() / class_counts).pow(CLASS_WEIGHT_POWER)
class_weights = class_weights / class_weights.mean()
class_weight_tensor = torch.ones(NUM_CLASSES + 1, dtype=torch.float32)
for class_id, weight in class_weights.items():
    class_weight_tensor[int(class_id)] = float(weight)

rare_class_ids = {
    class_id for class_id, label_name in label_map.items() if label_name in RARE_CLASS_NAMES
}

print(f"Annotations: {len(df):,}")
print(f"Images with boxes: {df['image_key'].nunique():,}")
print(f"Classes: {NUM_CLASSES}")
display(
    pd.DataFrame(
        {
            "label_name": pd.Series(label_map),
            "boxes": class_counts,
            "class_weight": class_weights.round(3),
            "rare_augmented": pd.Series({class_id: class_id in rare_class_ids for class_id in label_map}),
        }
    )
)
label_map

Annotations: 15,480
Images with boxes: 4,496
Classes: 9


,label_name,boxes,class_weight,rare_augmented
1,Cardiomegaly,2378,0.603,False
2,Pleural thickening,4056,0.462,False
3,Pulmonary fibrosis,3364,0.507,False
4,Pleural effusion,1855,0.683,False
5,Nodule/Mass,1865,0.681,False
6,Infiltration,1002,0.929,False
7,Consolidation,434,1.412,True
8,Atelectasis,331,1.617,True
9,Pneumothorax,195,2.106,True


{1: 'Cardiomegaly',
 2: 'Pleural thickening',
 3: 'Pulmonary fibrosis',
 4: 'Pleural effusion',
 5: 'Nodule/Mass',
 6: 'Infiltration',
 7: 'Consolidation',
 8: 'Atelectasis',
 9: 'Pneumothorax'}

In [7]:
# Notebook guide: match split annotation rows to local NIH/VinBig image files.
# The split is already saved in the CSV, so this cell only filters rows whose local image exists.

def image_path_for(source: str, image_id: str) -> Path | None:
    image_dir = IMAGE_DIRS.get(source)
    if image_dir is None:
        return None

    direct_candidate = image_dir / image_id
    if direct_candidate.exists():
        return direct_candidate

    stem = Path(image_id).stem
    for suffix in (".png", ".jpg", ".jpeg"):
        candidate = image_dir / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


image_lookup = df[["image_key", "source", "image_id"]].drop_duplicates()
image_paths = {
    row.image_key: image_path_for(row.source, row.image_id)
    for row in image_lookup.itertuples(index=False)
}
available_keys = [image_key for image_key, path in image_paths.items() if path is not None]
missing_count = len(image_paths) - len(available_keys)
if missing_count:
    print(f"Skipping {missing_count:,} source/image pairs without local image files.")

df = df[df["image_key"].isin(available_keys)].copy()
image_paths = {image_key: path for image_key, path in image_paths.items() if path is not None}
if df.empty:
    raise FileNotFoundError(f"No training images found under {IMAGE_DIRS}")

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()

print(f"Train images: {train_df['image_key'].nunique():,} / boxes: {len(train_df):,}")
print(f"Val images:   {val_df['image_key'].nunique():,} / boxes: {len(val_df):,}")

Train images: 3,822 / boxes: 11,954
Val images:   674 / boxes: 3,526


## 4. Dataset and Dataloaders

Build the PyTorch dataset and dataloaders. The dataset keeps every image, applies stronger augmentation only when an image contains a rare class, and uses a moderate weighted sampler to increase rare-class exposure without fully oversampling.

In [8]:
# Notebook guide: dataset and augmentation logic for EfficientDet training.
# Rare-class images receive stronger photometric augmentation and more frequent horizontal flips.
# Boxes are flipped with the image so coordinates remain aligned.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def has_rare_class(labels) -> bool:
    return any(int(label) in rare_class_ids for label in labels)


def augment_rare_image(image: Image.Image, boxes, rng: random.Random):
    if rng.random() < 0.65:
        image = F.hflip(image)
        boxes = boxes.copy()
        x_min = boxes[:, 0].copy()
        x_max = boxes[:, 2].copy()
        boxes[:, 0] = IMAGE_SIZE - x_max
        boxes[:, 2] = IMAGE_SIZE - x_min
    if rng.random() < 0.85:
        image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.75, 1.35))
    if rng.random() < 0.85:
        image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.80, 1.25))
    if rng.random() < 0.35:
        image = ImageEnhance.Sharpness(image).enhance(rng.uniform(0.75, 1.45))
    if rng.random() < 0.15:
        image = image.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.1, 0.6)))
    return image, boxes


def augment_common_image(image: Image.Image, boxes, rng: random.Random):
    if rng.random() < 0.25:
        image = F.hflip(image)
        boxes = boxes.copy()
        x_min = boxes[:, 0].copy()
        x_max = boxes[:, 2].copy()
        boxes[:, 0] = IMAGE_SIZE - x_max
        boxes[:, 2] = IMAGE_SIZE - x_min
    if rng.random() < 0.35:
        image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.90, 1.15))
    if rng.random() < 0.35:
        image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.90, 1.12))
    return image, boxes


class CXRDetectionDataset(Dataset):
    def __init__(self, annotations: pd.DataFrame, image_paths: dict[str, Path], image_size: int, augment: bool = False) -> None:
        self.annotations = annotations
        self.image_paths = image_paths
        self.image_size = image_size
        self.augment = augment
        self.image_keys = sorted(annotations["image_key"].unique())
        self.grouped = {image_key: group for image_key, group in annotations.groupby("image_key")}

    def __len__(self) -> int:
        return len(self.image_keys)

    def __getitem__(self, idx: int):
        image_key = self.image_keys[idx]
        image = Image.open(self.image_paths[image_key]).convert("RGB")
        original_w, original_h = image.size

        boxes = self.grouped[image_key][["x_min", "y_min", "x_max", "y_max"]].to_numpy(dtype="float32")
        if (original_w, original_h) != (self.image_size, self.image_size):
            scale_x = self.image_size / original_w
            scale_y = self.image_size / original_h
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y
            image = image.resize((self.image_size, self.image_size), Image.BILINEAR)

        labels = self.grouped[image_key]["target_class_id"].to_numpy(dtype="int64")
        if self.augment:
            rng = random.Random(f"{SEED}-{image_key}-{idx}-{random.random()}")
            if has_rare_class(labels):
                image, boxes = augment_rare_image(image, boxes, rng)
            else:
                image, boxes = augment_common_image(image, boxes, rng)

        image_tensor = F.to_tensor(image)
        image_tensor = F.normalize(image_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        # Effdet's anchor labeler expects boxes as [y_min, x_min, y_max, x_max].
        effdet_boxes = boxes[:, [1, 0, 3, 2]]

        target = {
            "bbox": torch.as_tensor(effdet_boxes, dtype=torch.float32),
            "cls": torch.as_tensor(labels, dtype=torch.int64),
            "img_size": torch.tensor([self.image_size, self.image_size], dtype=torch.float32),
            "img_scale": torch.tensor(1.0, dtype=torch.float32),
            "image_id": self.grouped[image_key]["image_id"].iloc[0],
            "image_key": image_key,
        }
        return image_tensor, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), {
        "bbox": [target["bbox"] for target in targets],
        "cls": [target["cls"] for target in targets],
        "img_size": torch.stack([target["img_size"] for target in targets]),
        "img_scale": torch.stack([target["img_scale"] for target in targets]),
        "image_id": [target["image_id"] for target in targets],
        "image_key": [target["image_key"] for target in targets],
    }

In [9]:
# Notebook guide: create dataloaders with moderate weighted sampling for the training set.
# `num_samples=len(train_dataset)` keeps epoch length stable while `replacement=True` lets rare images recur.

def image_sampling_weights(dataset: CXRDetectionDataset) -> torch.DoubleTensor:
    weights = []
    for image_key in dataset.image_keys:
        labels = dataset.grouped[image_key]["target_class_id"].astype(int)
        image_weight = class_weights.reindex(labels).max()
        weights.append(float(image_weight ** SAMPLER_WEIGHT_POWER))
    return torch.DoubleTensor(weights)


train_dataset = CXRDetectionDataset(train_df, image_paths, IMAGE_SIZE, augment=True)
val_dataset = CXRDetectionDataset(val_df, image_paths, IMAGE_SIZE, augment=False)
train_sampler = WeightedRandomSampler(
    weights=image_sampling_weights(train_dataset),
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

sample_images, sample_targets = next(iter(train_loader))
sample_images.shape, sample_targets["bbox"][0].shape, sample_targets["cls"][0]

(torch.Size([2, 3, 1024, 1024]), torch.Size([1, 4]), tensor([8]))

## 5. EfficientDet-D4 Model

Create the pretrained EfficientDet-D4 training bench. EfficientDet uses focal loss for dense detection; this cell also tries to set focal-loss parameters and class-weight fields when the installed `effdet` version exposes them.

In [10]:
# Notebook guide: build EfficientDet-D4 and configure available focal-loss/class-weight hooks.
# Different `effdet` versions expose loss internals differently, so this function checks defensively.
# Some installs store focal-loss settings in a read-only OmegaConf object, so failed mutations are skipped.

def try_setattr(obj, attr: str, value, label: str, configured: list[str], skipped: list[str]) -> None:
    try:
        setattr(obj, attr, value)
    except Exception as exc:
        skipped.append(f"{label}.{attr}: {type(exc).__name__}")
    else:
        configured.append(f"{label}.{attr}")


def configure_focal_loss_and_class_weights(model, weights: torch.Tensor) -> list[str]:
    configured = []
    skipped = []
    for obj_name, obj in [
        ("model.config", getattr(model, "config", None)),
        ("model.loss_fn", getattr(model, "loss_fn", None)),
        ("model.criterion", getattr(model, "criterion", None)),
    ]:
        if obj is None:
            continue
        for attr, value in [("gamma", FOCAL_LOSS_GAMMA), ("alpha", FOCAL_LOSS_ALPHA)]:
            if hasattr(obj, attr):
                try_setattr(obj, attr, value, obj_name, configured, skipped)
        for child_name in ("class_loss", "cls_loss", "classification_loss"):
            child = getattr(obj, child_name, None)
            if child is None:
                continue
            for attr, value in [("gamma", FOCAL_LOSS_GAMMA), ("alpha", FOCAL_LOSS_ALPHA)]:
                if hasattr(child, attr):
                    try_setattr(child, attr, value, f"{obj_name}.{child_name}", configured, skipped)
            for attr in ("class_weights", "class_weight"):
                if hasattr(child, attr):
                    try_setattr(child, attr, weights.to(DEVICE), f"{obj_name}.{child_name}", configured, skipped)
    if skipped:
        print("Skipped read-only or unsupported loss hooks:", skipped)
    return configured


model = create_model(
    "tf_efficientdet_d4",
    bench_task="train",
    num_classes=NUM_CLASSES,
    pretrained=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    bench_labeler=True,
)
model = model.to(DEVICE)
loss_configured = configure_focal_loss_and_class_weights(model, class_weight_tensor)
if loss_configured:
    print("Configured focal/class-weight hooks:", loss_configured)
else:
    print("Effdet focal loss is active by default; this install did not expose class-weight hooks.")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

Skipped read-only or unsupported loss hooks: ['model.config.gamma: ReadonlyConfigError', 'model.config.alpha: ReadonlyConfigError']
Configured focal/class-weight hooks: ['model.loss_fn.gamma', 'model.loss_fn.alpha']


## 6. Training Helpers

Define the reusable train/validation helpers. The training loss is multiplied by a batch-level class weight derived from the rarest class in each image, which makes class weighting explicit even when `effdet` does not expose native class-weight support.

In [11]:
# Notebook guide: helper functions for device movement, loss extraction, weighted loss, and checkpoints.
# Validation uses the raw detector loss; training uses the weighted loss multiplier for imbalance handling.

def move_targets_to_device(targets: dict, device: torch.device) -> dict:
    return {
        "bbox": [boxes.to(device) for boxes in targets["bbox"]],
        "cls": [labels.to(device) for labels in targets["cls"]],
        "img_size": targets["img_size"].to(device),
        "img_scale": targets["img_scale"].to(device),
        "image_id": targets.get("image_id", []),
        "image_key": targets.get("image_key", []),
    }


def loss_value(output):
    if isinstance(output, dict):
        return output["loss"]
    return output


def batch_class_weight(targets: dict) -> torch.Tensor:
    weights = class_weight_tensor.to(targets["cls"][0].device)
    image_weights = []
    for labels in targets["cls"]:
        labels = labels.clamp(min=0, max=len(weights) - 1)
        image_weights.append(weights[labels].max())
    return torch.stack(image_weights).mean()


IOU_THRESHOLDS = [round(0.50 + 0.05 * idx, 2) for idx in range(10)]
AREA_RANGES = {
    "small": (0, 32**2),
    "medium": (32**2, 96**2),
    "large": (96**2, float("inf")),
}


def effdet_yxyx_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    return boxes[:, [1, 0, 3, 2]]


def box_area_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    wh = (boxes[:, 2:] - boxes[:, :2]).clamp(min=0)
    return wh[:, 0] * wh[:, 1]


def pairwise_iou_xyxy(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((len(boxes1), len(boxes2)))
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    intersection = wh[:, :, 0] * wh[:, :, 1]
    union = box_area_xyxy(boxes1)[:, None] + box_area_xyxy(boxes2)[None, :] - intersection
    return intersection / union.clamp(min=1e-6)


def collect_detection_records(detections: torch.Tensor, targets: dict) -> tuple[list[dict], list[dict]]:
    predictions = []
    ground_truths = []
    detections = detections.detach().cpu()
    for batch_idx, image_key in enumerate(targets["image_key"]):
        gt_boxes = effdet_yxyx_to_xyxy(targets["bbox"][batch_idx].detach().cpu())
        gt_labels = targets["cls"][batch_idx].detach().cpu().long()
        gt_areas = box_area_xyxy(gt_boxes)
        for box, label, area in zip(gt_boxes, gt_labels, gt_areas):
            ground_truths.append(
                {"image_key": image_key, "class_id": int(label), "box": box, "area": float(area)}
            )

        valid_detections = detections[batch_idx][detections[batch_idx, :, 4] > 0]
        for det in valid_detections:
            predictions.append(
                {
                    "image_key": image_key,
                    "class_id": int(det[5].item()),
                    "box": det[:4],
                    "score": float(det[4].item()),
                }
            )
    return predictions, ground_truths


def average_precision(recalls: list[float], precisions: list[float]) -> float:
    if not recalls:
        return 0.0
    mrec = [0.0] + recalls + [1.0]
    mpre = [0.0] + precisions + [0.0]
    for idx in range(len(mpre) - 2, -1, -1):
        mpre[idx] = max(mpre[idx], mpre[idx + 1])
    return sum((mrec[idx] - mrec[idx - 1]) * mpre[idx] for idx in range(1, len(mrec)))


def evaluate_class_at_iou(predictions, ground_truths, class_id: int, iou_threshold: float, area_range=None, max_dets=100):
    class_gts = [gt for gt in ground_truths if gt["class_id"] == class_id]
    if area_range is not None:
        lo, hi = area_range
        class_gts = [gt for gt in class_gts if lo <= gt["area"] < hi]
    if not class_gts:
        return None

    gt_by_image = defaultdict(list)
    for gt in class_gts:
        gt_by_image[gt["image_key"]].append(gt["box"])
    matched = {image_key: set() for image_key in gt_by_image}

    pred_by_image = defaultdict(list)
    for pred in predictions:
        if pred["class_id"] == class_id and pred["image_key"] in gt_by_image:
            pred_by_image[pred["image_key"]].append(pred)
    kept_predictions = []
    for image_key, image_preds in pred_by_image.items():
        kept_predictions.extend(sorted(image_preds, key=lambda pred: pred["score"], reverse=True)[:max_dets])
    kept_predictions = sorted(kept_predictions, key=lambda pred: pred["score"], reverse=True)

    tp, fp = [], []
    for pred in kept_predictions:
        image_gts = gt_by_image[pred["image_key"]]
        gt_boxes = torch.stack(image_gts) if image_gts else torch.empty((0, 4))
        ious = pairwise_iou_xyxy(pred["box"].unsqueeze(0), gt_boxes).squeeze(0)
        best_iou, best_idx = (ious.max(0) if len(ious) else (torch.tensor(0.0), torch.tensor(-1)))
        best_idx = int(best_idx.item())
        if best_iou >= iou_threshold and best_idx not in matched[pred["image_key"]]:
            tp.append(1.0)
            fp.append(0.0)
            matched[pred["image_key"]].add(best_idx)
        else:
            tp.append(0.0)
            fp.append(1.0)

    if not tp:
        return {"ap": 0.0, "recall": 0.0}
    tp_cum = torch.tensor(tp).cumsum(0)
    fp_cum = torch.tensor(fp).cumsum(0)
    recalls = (tp_cum / len(class_gts)).tolist()
    precisions = (tp_cum / (tp_cum + fp_cum).clamp(min=1e-6)).tolist()
    return {"ap": average_precision(recalls, precisions), "recall": recalls[-1]}


def build_validation_metrics(predictions: list[dict], ground_truths: list[dict]) -> dict:
    per_class_ap = {}
    per_class_ar = {}
    ap_by_threshold = {threshold: [] for threshold in IOU_THRESHOLDS}
    ar_by_limit = {limit: [] for limit in (1, 10, 100)}
    ap_by_area = {name: [] for name in AREA_RANGES}
    ar_by_area = {name: [] for name in AREA_RANGES}

    for class_id in label_map:
        class_aps = []
        class_ars = []
        for threshold in IOU_THRESHOLDS:
            result = evaluate_class_at_iou(predictions, ground_truths, class_id, threshold, max_dets=100)
            if result is not None:
                class_aps.append(result["ap"])
                class_ars.append(result["recall"])
                ap_by_threshold[threshold].append(result["ap"])
        for limit in (1, 10, 100):
            limit_recalls = []
            for threshold in IOU_THRESHOLDS:
                result = evaluate_class_at_iou(predictions, ground_truths, class_id, threshold, max_dets=limit)
                if result is not None:
                    limit_recalls.append(result["recall"])
            if limit_recalls:
                ar_by_limit[limit].append(sum(limit_recalls) / len(limit_recalls))
        for area_name, area_range in AREA_RANGES.items():
            area_aps = []
            area_ars = []
            for threshold in IOU_THRESHOLDS:
                result = evaluate_class_at_iou(predictions, ground_truths, class_id, threshold, area_range=area_range)
                if result is not None:
                    area_aps.append(result["ap"])
                    area_ars.append(result["recall"])
            if area_aps:
                ap_by_area[area_name].append(sum(area_aps) / len(area_aps))
                ar_by_area[area_name].append(sum(area_ars) / len(area_ars))
        per_class_ap[class_id] = sum(class_aps) / len(class_aps) if class_aps else 0.0
        per_class_ar[class_id] = sum(class_ars) / len(class_ars) if class_ars else 0.0

    mean = lambda values: sum(values) / len(values) if values else 0.0
    return {
        "mAP": mean(list(per_class_ap.values())),
        "mAP50": mean(ap_by_threshold[0.50]),
        "mAP75": mean(ap_by_threshold[0.75]),
        "mAP_small": mean(ap_by_area["small"]),
        "mAP_medium": mean(ap_by_area["medium"]),
        "mAP_large": mean(ap_by_area["large"]),
        "AR1": mean(ar_by_limit[1]),
        "AR10": mean(ar_by_limit[10]),
        "AR100": mean(ar_by_limit[100]),
        "AR_small": mean(ar_by_area["small"]),
        "AR_medium": mean(ar_by_area["medium"]),
        "AR_large": mean(ar_by_area["large"]),
        "per_class_ap": per_class_ap,
        "per_class_ar": per_class_ar,
    }


def print_epoch_report(epoch: int, train_loss: float, val_loss: float, metrics: dict) -> None:
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Loss Details for Epoch {epoch}:")
    print(f"  - Training Loss: {train_loss:.6f}")
    print(f"  - Validation Loss: {val_loss:.6f}")
    print(f"  - Classification Loss: {metrics['classification_loss']:.6f}")
    print(f"  - Localization Loss: {metrics['localization_loss']:.6f}")
    print(f"Average Precision (AP) for Epoch {epoch}:")
    print(f"  - mAP: {metrics['mAP']:.4f}")
    print(f"  - mAP@.50: {metrics['mAP50']:.4f}")
    print(f"  - mAP@.75: {metrics['mAP75']:.4f}")
    print(f"  - mAP (small): {metrics['mAP_small']:.4f}")
    print(f"  - mAP (medium): {metrics['mAP_medium']:.4f}")
    print(f"  - mAP (large): {metrics['mAP_large']:.4f}")
    print(f"Average Recall (AR) for Epoch {epoch}:")
    print(f"  - AR@1: {metrics['AR1']:.4f}")
    print(f"  - AR@10: {metrics['AR10']:.4f}")
    print(f"  - AR@100: {metrics['AR100']:.4f}")
    print(f"  - AR (small): {metrics['AR_small']:.4f}")
    print(f"  - AR (medium): {metrics['AR_medium']:.4f}")
    print(f"  - AR (large): {metrics['AR_large']:.4f}")
    print(f"Per-Class mAP for Epoch {epoch}:")
    for class_id, label_name in label_map.items():
        print(f"  - {label_name}: {metrics['per_class_ap'].get(class_id, 0.0):.4f}")
    print(f"Per-Class AR (at 100 detections) for Epoch {epoch}:")
    for class_id, label_name in label_map.items():
        print(f"  - {label_name}: {metrics['per_class_ar'].get(class_id, 0.0):.4f}")


def train_one_epoch(epoch: int) -> float:
    model.train()
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = move_targets_to_device(targets, DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            output = model(images, targets)
            loss = loss_value(output) * batch_class_weight(targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(1, len(train_loader))


@torch.no_grad()
def validate_one_epoch(epoch: int) -> float:
    model.eval()
    total_loss = 0.0
    total_class_loss = 0.0
    total_box_loss = 0.0
    all_predictions = []
    all_ground_truths = []
    progress = tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = move_targets_to_device(targets, DEVICE)
        output = model(images, targets)
        loss = loss_value(output)
        total_loss += loss.item()
        if isinstance(output, dict):
            total_class_loss += float(output.get("class_loss", torch.tensor(0.0)).item())
            total_box_loss += float(output.get("box_loss", torch.tensor(0.0)).item())
            if "detections" in output:
                predictions, ground_truths = collect_detection_records(output["detections"], targets)
                all_predictions.extend(predictions)
                all_ground_truths.extend(ground_truths)
        progress.set_postfix(loss=f"{loss.item():.4f}")

    metrics = build_validation_metrics(all_predictions, all_ground_truths)
    metrics["classification_loss"] = total_class_loss / max(1, len(val_loader))
    metrics["localization_loss"] = total_box_loss / max(1, len(val_loader))
    return total_loss / max(1, len(val_loader)), metrics


def save_checkpoint(path: Path, epoch: int, val_loss: float) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_loss": val_loss,
            "label_map": label_map,
            "image_size": IMAGE_SIZE,
            "model_name": "tf_efficientdet_d4",
            "class_weights": class_weights.to_dict(),
            "rare_class_names": sorted(RARE_CLASS_NAMES),
            "focal_loss_gamma": FOCAL_LOSS_GAMMA,
            "focal_loss_alpha": FOCAL_LOSS_ALPHA,
        },
        path,
    )

## 7. Train

Run the training loop, write the best and latest checkpoints, and log configuration/metrics/artifacts to MLflow. Checkpoints include the label map and imbalance settings so inference/evaluation can trace how the model was trained.

In [12]:
# Notebook guide: run the full training loop and track the experiment in local MLflow.
# The best checkpoint is selected by validation loss; the last checkpoint captures the final epoch state.

mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").as_uri())
mlflow.set_experiment("cxraide-efficientdet-d4")

best_val_loss = float("inf")
best_checkpoint = OUTPUT_DIR / "best_tf_efficientdet_d4.pt"
last_checkpoint = OUTPUT_DIR / "last_tf_efficientdet_d4.pt"

with mlflow.start_run(run_name="tf_efficientdet_d4_1024"):
    mlflow.log_params(
        {
            "model": "tf_efficientdet_d4",
            "image_size": IMAGE_SIZE,
            "num_classes": NUM_CLASSES,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "train_images": train_df["image_key"].nunique(),
            "val_images": val_df["image_key"].nunique(),
            "rare_class_names": ",".join(sorted(RARE_CLASS_NAMES)),
            "class_weight_power": CLASS_WEIGHT_POWER,
            "sampler_weight_power": SAMPLER_WEIGHT_POWER,
            "focal_loss_gamma": FOCAL_LOSS_GAMMA,
            "focal_loss_alpha": FOCAL_LOSS_ALPHA,
        }
    )

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(epoch)
        val_loss, val_metrics = validate_one_epoch(epoch)
        scheduler.step()

        mlflow_metrics = {
            "train_loss": train_loss,
            "val_loss": val_loss,
            "classification_loss": val_metrics["classification_loss"],
            "localization_loss": val_metrics["localization_loss"],
            "mAP": val_metrics["mAP"],
            "mAP50": val_metrics["mAP50"],
            "mAP75": val_metrics["mAP75"],
            "AR100": val_metrics["AR100"],
        }
        for class_id, label_name in label_map.items():
            metric_name = label_name.lower().replace("/", "_").replace(" ", "_")
            mlflow_metrics[f"mAP_{metric_name}"] = val_metrics["per_class_ap"].get(class_id, 0.0)
            mlflow_metrics[f"AR100_{metric_name}"] = val_metrics["per_class_ar"].get(class_id, 0.0)
        mlflow.log_metrics(mlflow_metrics, step=epoch)
        save_checkpoint(last_checkpoint, epoch, val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(best_checkpoint, epoch, val_loss)

        print_epoch_report(epoch, train_loss, val_loss, val_metrics)
        print(f"Best validation loss so far: {best_val_loss:.6f}")

    mlflow.log_artifact(str(best_checkpoint))
    mlflow.log_artifact(str(last_checkpoint))

print(f"Best checkpoint: {best_checkpoint}")

d:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 1 train:   0%|          | 0/1911 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Inference Bench Sanity Check

Use this optional cell after training to load the best checkpoint into a prediction bench and confirm that inference returns detections for validation images. Keep it commented until a checkpoint exists.

In [ ]:
# Notebook guide: optional smoke test for a trained checkpoint.
# Uncomment after training to verify that the saved weights can be loaded for prediction.

# checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
# infer_model = create_model(
#     "tf_efficientdet_d4",
#     bench_task="predict",
#     num_classes=NUM_CLASSES,
#     pretrained=False,
#     image_size=(IMAGE_SIZE, IMAGE_SIZE),
# )
# infer_model.load_state_dict(checkpoint["model_state_dict"], strict=False)
# infer_model = infer_model.to(DEVICE).eval()
# images, targets = next(iter(val_loader))
# with torch.no_grad():
#     detections = infer_model(images.to(DEVICE))
# detections.shape